# SAIGE-GENE SET-BASED - FIT NULL MODEL 
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio are all set to TRUE. This script uses a sparse GRM, 5,000 randomly selected variants across specified MAC categories, and a phenotype file with covariates to fit the null model. Multiple case-to-control matched cohorts were used, with different matching ratios to generate distinct datasets, including 1:5, 1:10, 1:50, 1:100, and 1:300. The resulting null models are deposited in the workspace bucket.

In [ ]:
# Load packages 
import os
# Set environment variable for workspace bucket 
my_bucket = os.getenv('WORKSPACE_BUCKET')
# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Loop through case:control ratios 
    for ratio in 5 10 50 100 300; do 
    echo "Processing matched ratio: ${ratio}"
    
    # --sparseGRMFile : sparse GRM defining genetic relatedness for random effects in GLMM
    # --sparseGRMSampleIDFile : sample IDs matching GRM rows/columns to phenotype/genotype samples
    # --plinkFile : PLINK genotype subset used for estimating variance ratios across MAC categories
    # --phenoFile : phenotype file containing case/control labels and covariates for current cohort
    # --useSparseGRMtoFitNULL : use sparse GRM to fit null GLMM efficiently
    # --useSparseGRMforVarRatio : use sparse GRM when estimating variance ratios
    # --IsOverwriteVarianceRatioFile : overwrite existing variance ratio files if present
    # --isCateVarianceRatio : compute variance ratios stratified by MAC categories
    # --phenoCol : phenotype/outcome column used in GLMM model
    # --traitType : specifies binary trait model (case/control)
    # --covarColList : fixed-effect covariates included in the model (age, sex, PCs 1–10)
    # --qCovarColList : categorical covariates treated as qualitative factors (sex)
    # --sampleIDColinphenoFile : column linking phenotype rows to sample IDs
    # --outputPrefix : prefix for all SAIGE-GENE step 1 output files
    
    step1_fitNULLGLMM.R \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --plinkFile="${variance_files}/${pop}_array.5000.forCate_vr" \
        --phenoFile="${phenotype}/${pop}_matched_ratio${ratio}_cohort.txt" \
        --useSparseGRMtoFitNULL=TRUE \
        --useSparseGRMforVarRatio=TRUE \
        --IsOverwriteVarianceRatioFile=TRUE \
        --isCateVarianceRatio=TRUE \
        --phenoCol=aml_biosample_age \
        --traitType=binary \
        --covarColList=age,sex,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
        --qCovarColList=sex \
        --sampleIDColinphenoFile=person_id \
        --outputPrefix="${output}/${pop}_matched_ratio${ratio}_null_glmm" 
    done
done


In [ ]:
# Save script to workspace bucket 
!gsutil cp saige_gene_step01.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/saige_gene_step01.sh 

# SUBMIT COMMAND 

In [ ]:
%%bash --out saige_gene_step01

# Source bash script 
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

# --image : Docker image containing SAIGE (v1.5.0.2) and all required dependencies
# --boot-disk-size : size (GB) of VM boot disk used for system and runtime dependencies
# --disk-size : size (GB) of attached persistent disk used for input/output data
# --logging : cloud storage path where job logs (stdout/stderr) are written
# --input-recursive phenotype : input directory containing phenotype/cohort files (recursively mounted)
# --input-recursive grm : input directory containing sparse GRM files for relatedness correction
# --input-recursive variance_files : input directory containing PLINK genotype subsets for variance ratio estimation
# --output-recursive output : output directory for SAIGE-GENE step 1 results (null models + variance ratios)
# --env output : environment variable passed into container specifying output path
# --script : path to bash script executed inside container (SAIGE-GENE step 1 pipeline)

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 50 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/matched/cohort/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/saige_gene_step01.sh"
